## 1. Importação das Bibliotecas

In [ ]:
# Instala o pacote watermark
!pip install -q -U watermark

In [ ]:
# Importa as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
%reload_ext watermark
%watermark -a "Valter de Pauli Netto"

In [ ]:
watermark --iversions

In [ ]:
# Configurando o estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Geração dos Dados Sintéticos com Pandas e Numpy

In [ ]:
# --- Geração dos Dados Fictícios Coerentes ---
print("\nGerando o conjunto de dados fictícios...")

# Define a semente para resultados reproduzíveis
np.random.seed(42)

# Criando um dicionário de dados
data = {
        
    'ID_Pedido' : range(1001, 1101),
    'Data_Compra' : pd.to_datetime(pd.date_range(start = '2026-03-06', periods = 100, freq = 'D')) - pd.to_timedelta(np.random.randint(0, 30, size = 100), unit = 'd'),
    'Cliente_ID' : np.random.randint(100, 150, size = 100),
    'Produto' : np.random.choice(['SmartPhone', 'Notebook', 'Fone de Ouvido', 'Mouse', 'Teclado Mecânico'], size = 100),
    'Categoria' : ['Eletrônicos', 'Eletrônicos', 'Acessórios', 'Acessórios', 'Acessórios'] * 20,
    'Quantidade' : np.random.randint(1, 5, size = 100),
    'Preco_Unitario' : [5999.90, 8500.00, 799.50, 2100.00, 850.00] * 20,
    'Status_Entrega' : np.random.choice(['Entregue', 'Pendente', 'Cancelado'], size = 100, p = [0.8, 0.15, 0.05])
}

# Criando o dataframe a partir do dicionário
df_vendas = pd.DataFrame(data)

# Introduzindo Problemas nos Dados
print("\nIntroduzindo Problemas nos dados para a limpeza...\n")

# 1. Valores Ausentes (NaN)
df_vendas.loc[5:10, 'Quantidade'] = np.nan
df_vendas.loc[20:22, 'Status_Entrega'] = np.nan
df_vendas.loc[30, 'Cliente_ID'] = np.nan

# 2. Dados Duplicados
df_vendas = pd.concat([df_vendas, df_vendas.head(3)], ignore_index = True)

# 3. Tipos de Dados Incorretos
df_vendas['Preco_Unitario'] = df_vendas['Preco_Unitario'].astype(str)
df_vendas.loc[15, 'Preco_Unitario'] = 'valor_invalido'
df_vendas['Cliente_ID'] = df_vendas['Cliente_ID'].astype(str)
# Simulando um erro de digitação

# 4. Outliers
df_vendas.loc[50, 'Quantidade'] = 50
# Valor fora do padrão

print("Dados gerados com sucesso!\n")

In [ ]:
# Primeiras linhas
df_vendas.head()

In [ ]:
# Últimas linhas
df_vendas.tail()

## 3. Análise Exploratória de Dados (EDA)

In [ ]:
# Verificando as informações gerais do DataFrame
print("\n--- Informações Gerais do DataFrame (df_vendas.info()) ---\n")
df_vendas.info()

In [ ]:
print("\n--- Verificando valores ausentes ---\n")
print(df_vendas.isna().sum())

In [ ]:
print("\n--- Verificando a presença de registros duplicados ---\n")
print(f"Números de linhas duplicadas: {df_vendas.duplicated().sum()}")

In [ ]:
print("\n--- Estatísticas descritivas para colunas numéricas ---\n")
print(df_vendas.describe())

In [ ]:
print("\n--- Estatísticas descritivas para colunas categóricas ---\n")
print(df_vendas.describe(include = [object]))

In [ ]:
# Verificando as informações gerais do DataFrame
print("\n--- Tipos de dados ---\n")
df_vendas.dtypes

## 4. Limpeza e Pré-Processamento dos Dados

In [ ]:
# Copiando o DataFrame para manter o original intacto
df_limpo = df_vendas.copy()

In [ ]:
# 1. Corrigindo Tipos de Dados, convertendo 'Preco_Unitario' em dado numérico
print("Corrigindo tipos de dados...")
df_limpo['Preco_Unitario'] = pd.to_numeric(df_limpo['Preco_Unitario'], errors = 'coerce')
# errors = 'coerce' transformará valores inválidos em NaN

In [ ]:
# Convertendo 'Cliente_ID' para numérico
df_limpo['Cliente_ID'] = pd.to_numeric(df_limpo['Cliente_ID'], errors = 'coerce').astype('Int64')
# Int64 para permitir NaN

In [ ]:
df_limpo.dtypes

In [ ]:
# 2. Tratando Valores Ausentes (NaN)
print("Tratando valores ausentes...")
mediana_qtd = df_limpo['Quantidade'].median()
df_limpo.fillna({'Quantidade': mediana_qtd}, inplace = True)
# 'Quantidade' é preenchida por mediana, por ser mais robusta a outliers

In [ ]:
# Para 'Status_Entrega', é preenchido com o valor mais frequente (moda)
moda_status = df_limpo['Status_Entrega'].mode()[0]
df_limpo['Status_Entrega'] = df_limpo['Status_Entrega'].fillna(moda_status)

In [ ]:
# A melhor alternativa para 'Preco_Unitario' e 'Cliente_ID' é remover as linhas, ja que não se pode inferir esses dados, devido terem sido gerados por erro ou falta de informação
df_limpo.dropna(subset = ['Preco_Unitario', 'Cliente_ID'], inplace = True)

In [ ]:
# 3. Removendo Duplicatas
print("Removendo duplicatas...")
df_limpo.drop_duplicates(inplace = True)

In [ ]:
# Criando um mapeamento correto
mapa_categorias = {
    'SmartPhone': 'Eletrônicos',
    'Notebook': 'Eletrônicos',
    'Fone de Ouvido': 'Acessórios',
    'Mouse': 'Acessórios',
    'Teclado Mecânico': 'Acessórios'
}

# Aplicando o mapeamento para corrigir a coluna Categoria baseada no Produto
df_limpo['Categoria'] = df_limpo['Produto'].map(mapa_categorias)

In [ ]:
# 4. Tratando Outliers
print("Tratando outliers...")
sns.boxplot(x = df_limpo['Quantidade'])
plt.title('Boxplot de Quantidade - Antes do tratamento')
plt.show()

In [ ]:
# Removendo valores de 'Quantidade' que estão além de 3 desvios padrão da média
limite_superior = df_limpo['Quantidade'].mean() + 3 * df_limpo['Quantidade'].std()
df_limpo = df_limpo[df_limpo['Quantidade'] < limite_superior]

In [ ]:
# Verificando o resultado
sns.boxplot(x = df_limpo['Quantidade'])
plt.title('Boxplot de Quantidade - depois do tratamento')
plt.show()

In [ ]:
# Verificação Final
print("\n--- Verificação Final Pós-Limpeza ---\n")
df_limpo.info()
print("\nValores ausentes restantes:\n", df_limpo.isna().sum())
print(f"\nLinhas duplicadas restantes: {df_limpo.duplicated().sum()}")

## 5. Engenharia de Atributos e Extração de Insights

In [ ]:
df_limpo.head()

In [ ]:
# Criando uma nova coluna 'Total_Vendas'
df_limpo['Total_Vendas'] = df_limpo['Quantidade'] * df_limpo['Preco_Unitario']

In [ ]:
df_limpo.head()

In [ ]:
# Total da Receita
receita_total = df_limpo['Total_Vendas'].sum()
print(f"A receita total da loja foi de: R$ {receita_total:,.2f}")

In [ ]:
# Receita total por categoria
receita_total_categoria = df_limpo.groupby('Categoria')['Total_Vendas'].sum().sort_values(ascending = False)
print("\n--- Receita Total por Categoria: ---\n")
print(receita_total_categoria)

In [ ]:
# Produto mais vendido em quantidade
produto_mais_vendido = df_limpo.groupby('Produto')['Quantidade'].sum().sort_values(ascending = False)
print("\n--- Total de Unidades Vendidas por Produto ---\n")
print(produto_mais_vendido)

In [ ]:
# Análise de vendas por dias
vendas_por_dia = df_limpo.set_index('Data_Compra').resample('D')['Total_Vendas'].sum()
print("\n--- Resumo de Vendas por Dia (Primeiros 5 dias) ---\n")
print(vendas_por_dia.head())

## 6. Visualização dos Dados e Análise Gráfica

In [ ]:
# Gráfico 1 -> Receita por Categoria
receita_total_categoria.plot(kind = 'bar', color = 'skyblue')
plt.title('Receita Total Por Categoria de Produto')
plt.ylabel('Receita (R$)')
plt.xlabel('Categoria')
plt.xticks(rotation = 0)
plt.show()

In [ ]:
# Gráfico 2 -> Quantidade Vendida por Produto
produto_mais_vendido.plot(kind = 'barh', color = 'salmon')
plt.title('Quantidade de Unidades Vendidas Por Produto')
plt.ylabel('Produto')
plt.xlabel('Quantidade Vendida')
plt.gca().invert_yaxis() # Inverte o eixo para o maior valor ficar no topo
plt.show()

In [ ]:
# Gráfico 3 -> Tendência de Vendas ao Longo do Tempo
vendas_por_dia.plot(kind = 'line', marker = '.', linestyle = '-')
plt.title('Tendência de Vendas Diárias')
plt.ylabel('Receita (R$)')
plt.xlabel('Data da Compra')
plt.grid(True)
plt.show()

In [ ]:
# Gráfico 4.1 -> Distribuição do Status de Entrega

# Conta quantas vezes aparece cada status de entrega
status_counts = df_limpo['Status_Entrega'].value_counts()

plt.pie(
    status_counts,                 # Valores numéricos para cada fatia 
    labels = status_counts.index,  # Rótulos de cada fatia 
    autopct = '%1.1f%%',           # Mostra o percentual em cada fatia com 1 casa decimal 
    startangle = 180,              # Ângulo inicial para "girar" o gráfico e escolher onde começa a primeira fatia
    colors = ['green',        # Cor da primeira fatia
              'orange',            # Cor da segunda fatia
              'lightcoral']        # Cor da terceira fatia
)

plt.title('\nDistribuição do Status de Entrega')  
plt.show()                                         

In [ ]:
# Gráfico 4.2 -> Distribuição do Status de Entrega no formato 3D

# Conta quantas vezes aparece cada status de entrega
status_counts = df_limpo['Status_Entrega'].value_counts()

# Descobre a posição (índice) da fatia com maior valor para destacá-la
maior_idx = status_counts.argmax()

# Cria a lista explode: desloca 0.1 para a maior fatia e 0 para as outras
explode = [0.1 if i == maior_idx else 0 for i in range(len(status_counts))]

# Define o tamanho da figura (6x6 polegadas)
plt.figure(figsize = (6,6))

plt.pie(
    status_counts,                 # Valores numéricos para cada fatia (quantidade de cada status)
    labels = status_counts.index,  # Rótulos de cada fatia (nomes dos status)
    autopct = '%1.1f%%',           # Mostra o percentual em cada fatia com 1 casa decimal 
    startangle = 180,              # Ângulo inicial para "girar" o gráfico e definir onde começa a primeira fatia
    colors = ['green',        # Cor da primeira fatia
              'orange',            # Cor da segunda fatia
              'red'],       # Cor da terceira fatia
    explode = explode,             # Desloca a maior fatia para destacá-la visualmente
    shadow = True                  # Adiciona sombra para criar um efeito 3D simples
)

plt.title('\nDistribuição do Status de Entrega\n')  # Define o título do gráfico
plt.axis('equal')                                   # Mantém o formato circular (sem deformações)
plt.show()                                          # Exibe o gráfico

In [ ]:
# Gráfico 4.3 -> Distribuição dos Status de Entrega com gráfico interativo usando o Plotly

# Importa o pacote Plotly Express para gráficos interativos
import plotly.express as px

# Cria o gráfico de pizza interativo
dsa_fig = px.pie(
    values = status_counts,        # Valores numéricos para cada fatia 
    names = status_counts.index,   # Rótulos de cada fatia 
    hole = 0,                      # Define o tamanho do "furo" no centro (0 = pizza completa, > 0 cria gráfico do tipo donut)
    title = 'Distribuição do Status de Entrega' 
)

# Ajusta o destaque das fatias 
dsa_fig.update_traces(
    pull = [0.05 if i == maior_idx else 0 for i in range(len(status_counts))]
    # Cria uma lista onde a maior fatia é deslocada 0.05 e as outras ficam sem deslocamento
)

dsa_fig.show()